# 03 - Prepare non-NBA inputs

Cohorts genotyped outside the GP2 release: they are QCed and imputed by
`00_genotools_qc_imputation.ipynb` first, then prepared here one cohort at a
time. Each cohort becomes its own dataset in the meta-analysis.

All of these cohorts are European, so `ancestry` is fixed to EUR rather than from the master key.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

from gwas import config as cfg
from gwas import prep

pd.options.mode.chained_assignment = None

cfg.describe()

In [ ]:
# Mount the release and working buckets with gcsfuse (run once per VM).
for cmd in cfg.mount_commands():
    print(cmd)

## Parameters

In [ ]:
ANALYSIS = 'CS_disc'    # 'CS_disc' or 'SURV'
COHORT = 'PARKFIT'      # DATATOP, OPDC, PARKFIT, PROBAND, SCOPA
ANCESTRY = 'EUR'

# Some cohorts are named differently in the clinical dataset than in the
# genotype files; the cohort name used here is the genotype-side one.
CLINICAL_COHORT = {'SCOPA': 'PROPARK'}.get(COHORT, COHORT)

# Folder holding the cohort's QCed, normalised pgen files (notebook 00).
GENO_DIR = cfg.WORK_MOUNT / cfg.PREFIX / 'non_NBA' / COHORT / 'PLINK'

# PROBAND genotypes carry their own IDs; the map is written by notebook 00.
ID_MAP_PATH = None   # e.g. GENO_DIR.parent / f'{COHORT}_ID_map.csv'

# True when the cohort's psam uses the doubled "<IID>_<IID>" form.
DOUBLE_ID = COHORT != 'PROBAND'

OUT_DIR = cfg.prep_path(*cfg.prep_parts(cohort=COHORT), ANALYSIS)
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(OUT_DIR)

## Clinical data

In [ ]:
clinical = pd.read_csv(cfg.clinical_path(ANALYSIS, source='non_NBA'), low_memory=False)
clinical = clinical.rename(columns={'id': 'GP2ID'}).drop_duplicates()
display(clinical.cohort.value_counts())

subset = clinical[clinical.cohort == CLINICAL_COHORT].copy()
print(f'{len(subset)} rows, {subset.GP2ID.nunique()} samples in {COHORT}')

# These fields come from the master key for release cohorts; here they are
# known properties of the cohort.
subset['nba_label'] = ANCESTRY
subset['GP2_phenotype'] = 'PD'
subset['study_type'] = 'Case(/Control)'

subset = prep.check_required_fields(subset, ANALYSIS)
display(subset.head())

## Sample IDs

Map the clinical IDs onto the IDs in the cohort's PLINK files, then keep only
samples present in both.

In [ ]:
id_map = pd.read_csv(ID_MAP_PATH) if ID_MAP_PATH else None
subset = prep.cohort_sample_ids(subset, COHORT, id_map=id_map, double_id=DOUBLE_ID)

psam = pd.read_csv(GENO_DIR / f'{COHORT}_chr22_norm.psam', sep='\t')
keep = psam[psam['#IID'].isin(subset['#IID'])]
print(f'{len(keep)} of {subset["#IID"].nunique()} samples have genotypes')

keep[['#IID']].to_csv(OUT_DIR / f'non_NBA_{COHORT}_keep_samples.txt',
                      sep='\t', index=False)

subset = subset.merge(psam[['#IID']], on='#IID', how='inner')
display(subset.head())

## Covariates and phenotypes

Non-NBA psam files carry a single `#IID` column rather than the `#FID` / `IID`
pair used by the release genotypes.

In [ ]:
ID_COLS = ('#IID',)

covar = prep.format_covariates(subset, ANALYSIS, id_cols=ID_COLS)
covar.to_csv(OUT_DIR / cfg.covar_file(ANALYSIS, COHORT), sep='\t', index=False)

print(f'{len(covar)} covariate rows')
display(covar.head())

bad_dd = covar[covar.disease_duration < 0]
if len(bad_dd):
    print(f'WARNING: {len(bad_dd)} samples have a negative disease duration')
    display(bad_dd)

In [ ]:
pheno = prep.format_phenotypes(subset, ANALYSIS, id_cols=ID_COLS)
pheno.to_csv(OUT_DIR / cfg.pheno_file(ANALYSIS, COHORT), sep='\t', index=False)
display(pheno.head())

## Fixes

Survival runs drop events recorded at day 0. Samples with a non-positive
disease duration are dropped from both analyses: onset after baseline means
the clinical timeline is wrong, and the model cannot use it.

In [ ]:
covar_path = OUT_DIR / cfg.covar_file(ANALYSIS, COHORT)
pheno_path = OUT_DIR / cfg.pheno_file(ANALYSIS, COHORT)

for path in (covar_path, pheno_path):
    backup = path.with_name(path.stem + '_PRE_FIX.tsv')
    backup.write_bytes(path.read_bytes())
    print(f'backed up {backup.name}')

covar = pd.read_csv(covar_path, sep='\t')
pheno = pd.read_csv(pheno_path, sep='\t')

n_before = len(covar)
covar = covar[covar.disease_duration > 0]
pheno = pheno[pheno['#IID'].isin(covar['#IID'])]
print(f'dropped {n_before - len(covar)} samples with disease_duration <= 0')

if ANALYSIS == 'SURV':
    covar, pheno = prep.drop_zero_time_events(covar, pheno, id_col='#IID')

covar.to_csv(covar_path, sep='\t', index=False)
pheno.to_csv(pheno_path, sep='\t', index=False)
print(f'{len(covar)} covariate rows, {len(pheno)} phenotype rows')

## Upload

In [ ]:
dest = cfg.prep_url(*cfg.prep_parts(cohort=COHORT))
print(f'gsutil -m cp -r {OUT_DIR} {dest}/')

The cohort's genotypes are already normalised and subset pgen files in the bucket, so
there is no subsetting step here that is needed with release-ready files. The pipeline points straight to them
from `04_launch_gwas.ipynb` (the `GENO_ROOT` placeholder).